In [1]:
import pandas as pd
import numpy as np

In [2]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

In [3]:
public = pd.read_csv('public_trip_data.csv')
private = pd.read_csv('private_trip_data.csv')

In [4]:
public.head()

,TripID,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingType,ShippingTypeDescription,Purpose,OutOfPolicy,EntitiyCode,EmployeeNumber,BusinessUnit,HotelNights,NetCosts,Departure_CO2e,Return_CO2e,Hotel_CO2e,Spend_CO2e,TotalCO2e,HighCarbon
0,1,CN,Beijing,IN,New Delhi,12,Business Class Flight,Customer Visit,No,9000,M1000-M5008,Services,1,900,1565.121857,1565.121857,58.9,1472.279257,3189.143714,0
1,3,US,New York,MX,Mexico City,11,First Class Flight,Customer Visit,Yes,6000,M1000-M5030,Sales,2,3600,743.295848,743.295848,38.6,2921.504761,1525.191696,0
2,6,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Conference/Exhibition,No,9000,M1000-M5069,Services,1,450,1053.713528,1053.713528,51.4,743.646219,2158.827056,0
3,7,DE,Berlin,FR,Paris,12,Business Class Flight,Customer Visit,No,10000,M1000-M5095,Executive Management,1,3450,196.262703,196.262703,6.7,3343.160260,399.225407,0
4,8,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Internal Business Trip,No,7000,M1000-M5097,Marketing,2,1500,1053.713528,1053.713528,102.8,2478.820729,2210.227056,0


In [5]:
df = public.copy()
private_initial = private.copy()

In [6]:
print(public["HighCarbon"].value_counts())
print(public["HighCarbon"].value_counts(normalize=True))

HighCarbon
0    48963
1    16326
Name: count, dtype: int64
HighCarbon
0    0.749943
1    0.250057
Name: proportion, dtype: float64


In [7]:
#missing
df.isna().sum()

TripID                      0
DepartureLocationCountry    0
DepartureLocationCity       0
ArrivalLocationCountry      0
ArrivalLocationCity         0
ShippingType                0
ShippingTypeDescription     0
Purpose                     0
OutOfPolicy                 0
EntitiyCode                 0
EmployeeNumber              0
BusinessUnit                0
HotelNights                 0
NetCosts                    0
Departure_CO2e              0
Return_CO2e                 0
Hotel_CO2e                  0
Spend_CO2e                  0
TotalCO2e                   0
HighCarbon                  0
dtype: int64

In [32]:
#feature engineering
#Domestic vs International
public["InternationalTrip"] = (
    public["DepartureLocationCountry"] != public["ArrivalLocationCountry"]
).astype(int)

private["InternationalTrip"] = (
    private["DepartureLocationCountry"] != private["ArrivalLocationCountry"]
).astype(int)


# Country Pair
public["CountryPair"] = (
    public["DepartureLocationCountry"] + "_" +
    public["ArrivalLocationCountry"]
)
private["CountryPair"] = (
    private["DepartureLocationCountry"] + "_" +
    private["ArrivalLocationCountry"]
)


# City Pair
public["CityPair"] = (
    public["DepartureLocationCity"] + "_" +
    public["ArrivalLocationCity"]
)

private["CityPair"] = (
    private["DepartureLocationCity"] + "_" +
    private["ArrivalLocationCity"]
)

In [34]:
forbidden_cols = ['Departure_CO2e','Return_CO2e','Hotel_CO2e','Spend_CO2e','TotalCO2e','HighCarbon',    'TripID','EmployeeNumber'] #we just removing trip id cuz no use, and employee number not in private soooo


features = [col for col in public.columns if col not in forbidden_cols]
numericals = public.select_dtypes(include='number').columns.tolist()
num_cols = [col for col in numericals if col in features] 
cat_cols = [col for col in features if col not in num_cols]

public = public[features]
private = private[features]

In [35]:
print(cat_cols)

['DepartureLocationCountry', 'DepartureLocationCity', 'ArrivalLocationCountry', 'ArrivalLocationCity', 'ShippingTypeDescription', 'Purpose', 'OutOfPolicy', 'BusinessUnit', 'CountryPair', 'CityPair']


In [36]:
#prediction time

In [37]:
# from catboost import CatBoostClassifier
# from sklearn.model_selection import GridSearchCV

# model = CatBoostClassifier(verbose=0)

# param_grid = {
#     "iterations": [100, 200, 300],
#     "learning_rate": [0.01, 0.05, 0.1],
#     "depth": [3, 4, 5]
# }

# grid = GridSearchCV(
#     model,
#     param_grid,
#     cv=5,
#     scoring="roc_auc",
#     n_jobs=-1
# )

# grid.fit(
#     public[features],
#     public["HighCarbon"],
#     cat_features=cat_cols
# )

In [38]:
# print("Best Parameters:", grid.best_params_)
# print("Best CV Score:", grid.best_score_)
# # print("Best Model:", grid.best_estimator_)

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    public[features],
    df["HighCarbon"],
    test_size=0.2,
    random_state=42
)

model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=5,
    verbose=0
)

model.fit(X_train, y_train, cat_features=cat_cols)

probs = model.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, probs))

0.9993332339515739


In [41]:
# Final CatBoost model
cat_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=5,
    random_state=42,
    verbose=0
)

cat_model.fit(
    public[features],
    df["HighCarbon"],
    cat_features=cat_cols
)
probs = cat_model.predict_proba(private)[:, 1]

submission = pd.DataFrame({
    "TripID": private_initial["TripID"],
    "HighCarbon": probs
})
submission.to_csv("submission.csv", index=False)

In [42]:
submission.head()

,TripID,HighCarbon
0,2,0.999697
1,4,0.001090
2,5,0.878067
3,14,0.000029
4,15,0.000035
